# Example 5.5 / 1: Ranking Values in AMPL

Companion notebook for **5.5 Sorting Parameter Values** in the AMPL lab Documentation, section "Ranking Values in AMPL". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Ranking Values in AMPL

The example ranks products by margin. Two products tie and therefore receive the same rank.
`to_dict()` retrieves the values; Python's `sorted` then orders the report by rank and name.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                              # Products
param margin {PROD};                   # Margin of each product

param rank {p in PROD} integer >= 1 =  # Competition rank: ties share a rank
    1 + card({q in PROD: margin[q] > margin[p]});

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate", "sheet"]
ampl.param["margin"] = {"bands": 15, "coils": 22, "plate": 15, "sheet": 8}

In [5]:
# run.py
margins = ampl.param["margin"].to_dict()
ranks = ampl.param["rank"].to_dict()
for product in sorted(margins, key=lambda p: (ranks[p], p)):
    print(f"{int(ranks[product])}: {product} ({margins[product]:g})")

1: coils (22)
2: bands (15)
2: plate (15)
4: sheet (8)


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := bands coils plate sheet;
param margin := bands 15  coils 22  plate 15  sheet 8;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
for {r in 1..card(PROD), p in PROD: rank[p] = r} {
    printf "%d: %s (%g)\n", r, p, margin[p];
}

1: coils (22)
2: bands (15)
2: plate (15)
4: sheet (8)
